# Transformer evaluation and input/output

This notebook answers: **what changed, how was it measured, how well did it generalize, and what can we claim?** It checks saved evidence, recomputes metrics from saved per-frame predictions, compares every model/seed and plays verified examples.

Run all cells using **BPDR local Python** through `Run-Notebooks.ps1`, or select `.venv/Scripts/python.exe` in VS Code. Evaluation here is a reproducible reading/recomputation of completed studies; no fitting, parameter search or dataset extraction occurs.

Related: [training](03_transformer_training.ipynb), [data](01_input_validation_and_analysis.ipynb), [complete study record](02_data_review_and_paired_experiments.ipynb).

In [ ]:
# Locate the workspace before importing project modules (JupyterLab or VS Code).
import os, sys, importlib.util
from pathlib import Path
start = Path(os.environ.get('BPDR_COMPONENT_ROOT', Path.cwd())).resolve()
COMPONENT = next((p for p in (start, *start.parents)
    if p.name == 'bpdr_input' and (p/'pyproject.toml').is_file()), None)
if COMPONENT is None:
    raise RuntimeError('Open this notebook inside BPDR_Workspace, using Run-Notebooks.ps1.')
sys.path.insert(0, str(COMPONENT/'BPDR_Workspace/source'))
required = ['numpy', 'pandas', 'matplotlib', 'soundfile', 'torch', 'IPython']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Missing packages: '+', '.join(missing)+
        '. Select '+str(COMPONENT/'.venv/Scripts/python.exe')+' as the notebook kernel.')
print('Python:', sys.executable)
print('Workspace:', COMPONENT/'BPDR_Workspace')

In [ ]:
import json, hashlib, inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Audio, Code
from bpdr_input.reporting.notebook_evidence import load_index
from bpdr_input.experiments.evaluation.transfer_study import read, sha
INDEX, EVIDENCE = load_index(COMPONENT)
M = read(EVIDENCE['presentation_manifest'])
assert sha(EVIDENCE['presentation_manifest']) == INDEX['current_gate']['presentation_manifest_sha256']
for path, digest in M['source_hashes'].items():
    assert sha(path) == digest, path
release = EVIDENCE['presentation_manifest'].parent
assert sha(release/'model_card.json') == M['model_card_sha256']
CARD = read(release/'model_card.json')
working = COMPONENT/'BPDR_Workspace/selected_model'
working_card = read(working/'model.json')
CHECKPOINT = working/working_card['checkpoint']
assert sha(CHECKPOINT) == working_card['sha256'] == CARD['checkpoint_sha256']
SYNTH = {seed: read(Path(folder)/'evidence.json') for seed,folder in M['synthetic_runs'].items()}
ADAPT_DIR = EVIDENCE['presentation_adapt'].parent
ADAPT = read(EVIDENCE['presentation_adapt'])
assert sha(ADAPT_DIR/'declaration.json') == ADAPT['declaration_sha256']
print('Verified saved evidence and selected checkpoint:', sha(CHECKPOINT))
print('This notebook reads saved experiments; it does not fit or replace a model.')

## 1. Evaluation design and metric meanings

Four views share one aligned time grid. Let $P$ be reliable frames inside the nonzero injected shift, $O$ reliable frames outside it, $M$ the reliable mask, and $t=-e$ the compensating target.

| Metric | Calculation | Interpretation |
| --- | --- | --- |
| Correction MAE | Average C/D absolute error against $t$ on $P$ | Lower is better, in cents |
| Zero MAE | Absolute target magnitude on $P$ | Baseline if no correction is made |
| Correction gain | $1-\mathrm{MAE}/\mathrm{zeroMAE}$ | Error reduction, not classification accuracy |
| Paired response | Mean $(r_C-r_A)$ on $P$ divided by mean $t$ | Ideal 1; constant output gives 0 |
| Boundary sensitivity | Mean A–B and C–D absolute difference on $M$ | Lower means less dependence on changed boundaries |
| Original alteration | Mean absolute A/B correction on $M$ | Preservation of original input |
| Outside alteration | Mean absolute C/D correction on $O$ | Preservation outside the injected error |

Per-variant metrics are summarized by arithmetic mean except response, which uses the median. The real-source scorer averages original alteration across A/B and outside alteration across C/D; the older controlled-study scorer uses A and C respectively. We retain each recorded definition and do not silently replace historical values. These are dependent perturbation variants, not an independent participant success rate. The real-source engineering gate requires **each dataset** to have response ≥0.25, original alteration ≤1 cent and correction gain >0. Passing this numerical gate would still require musical/listening validation.

Development data is singer/artist-group separated but has supported several diagnostics; no untouched final-test result is claimed. Vocadito's performed reference is not proof of intended musical correctness, and Jamendo labels are model predictions.

In [ ]:
fine=COMPONENT/'runs/fine_pitch/66e3619e611241dd80d33a32578b73c6'
train_records=read(fine/'training_catalog.json')['records']
dev_records=read(fine/'development_catalog.json')['records']
ds=lambda r:r.get('dataset','vocadito' if r['phrase_id'].startswith('vocadito') else 'jamendo')
assert not {(ds(r),r['singer_id']) for r in train_records} & {(ds(r),r['singer_id']) for r in dev_records}
display(pd.DataFrame([{'dataset':d,'development_variants':sum(ds(r)==d for r in dev_records),
    'independent_group_proxies':len({r['singer_id'] for r in dev_records if ds(r)==d})}
    for d in ['vocadito','jamendo']]))
print('Final test accessed in recorded release:',M['final_test_accessed'])
assert not M['final_test_accessed']

## 2. Recompute one real evaluation case from saved predictions

This uses the **actual adapted BPDR per-frame output** and the matching saved reference mask/target. Targets are used only for scoring. The model did not receive them as acoustic inputs. The checks below compare our calculations with the recorded row, so the metric table is traceable to the saved predictions rather than manually entered scores.

In [ ]:
from bpdr_input.modeling.examples import read_pair
saved_rows=read(ADAPT_DIR/'bpdr/rows.json')
row=saved_rows[0];record=dev_records[row['index']]
assert row['phrase_id']==record['phrase_id'] and row['cents']==record['cents']
pair=read_pair(record,'engineering')
with np.load(ADAPT_DIR/'bpdr/predictions.npz') as archive:
    r={v:archive[f"{row['index']}_{v}"].copy() for v in 'ABCD'}
mask=pair['mask']; target=-pair['detuning']; P=mask&(pair['detuning']!=0); O=mask&(pair['detuning']==0)
computed=dict(correction_mae=float(np.mean([np.abs(r[v][P]-target[P]).mean() for v in 'CD'])),
    zero_mae=float(np.abs(target[P]).mean()),
    response_gain=float((r['C'][P]-r['A'][P]).mean()/target[P].mean()),
    boundary_sensitivity=float((np.abs(r['A'][mask]-r['B'][mask]).mean()+np.abs(r['C'][mask]-r['D'][mask]).mean())/2),
    unshifted_alteration=float(np.mean([np.abs(r[v][mask]).mean() for v in 'AB'])),
    outside_alteration=float(np.mean([np.abs(r[v][O]).mean() for v in 'CD'])))
for metric,value in computed.items(): np.testing.assert_allclose(value,row[metric],rtol=1e-5,atol=1e-5)
display(pd.DataFrame([{'metric':k,'recomputed':v,'recorded':row[k]} for k,v in computed.items()]))
fig,axes=plt.subplots(2,1,figsize=(11,6),sharex=True)
axes[0].plot(target,label='Known compensating target',color='#a44324')
axes[0].plot(r['C'],label='Predicted C correction');axes[0].set(ylabel='Cents',title='Real-source development case: '+row['phrase_id'])
axes[1].plot(r['C']-r['A'],label='Paired prediction change');axes[1].plot(target,label='Ideal response',ls=':')
axes[1].set(xlabel='Frame index',ylabel='Cents')
for ax in axes:ax.legend()
plt.tight_layout();plt.show()

## 3. All controlled training runs and matched comparisons

We retain every seed and condition. The augmentation control sees boundary metadata but has no BPDR response terms; the frame-only control does not see the boundary feature. A zero correction baseline exposes apparent stability that simply ignores detuning.

In [ ]:
rows=[dict(seed=seed,model=condition,**result['heldout'])
    for seed,evidence in SYNTH.items() for condition,result in evidence['results'].items()]
table=pd.DataFrame(rows)
display(table[['seed','model','records','correction_mae','zero_mae','median_response_gain','boundary_sensitivity','unshifted_alteration','outside_alteration']])
fig,axes=plt.subplots(1,2,figsize=(12,4))
for condition in ['augmentation_only','frame_only','bpdr']:
    part=table[table.model==condition]
    axes[0].plot(part.seed,part.correction_mae,'o-',label=condition)
    axes[1].plot(part.seed,part.median_response_gain,'o-',label=condition)
axes[0].axhline(table.zero_mae.iloc[0],ls=':',color='grey',label='Zero correction')
axes[1].axhline(1,ls=':',color='grey');axes[0].set(ylabel='Correction MAE (cents)',xlabel='Seed')
axes[1].set(ylabel='Paired response gain (ideal 1)',xlabel='Seed');axes[0].legend(fontsize=8)
plt.tight_layout();plt.show()
stats=read(EVIDENCE['presentation_statistics'])
display(pd.DataFrame([dict(seed=s,control=c,**value) for s,comparisons in stats['synthetic'].items() for c,value in comparisons.items()]))
print('Intervals resample six generated voice settings, not human participants; seeds stay separate.')

Seed 340 collapsed toward a constant response. Favorable generated-voice results in seeds 339/341 therefore do not establish stable training or a consistent BPDR advantage. Paired intervals use group-level comparisons and have limited independent groups.

## 4. Real Vocadito/Jamendo transfer, adaptation and preservation

Frozen transfer evaluated pretrained weights without updates. Adaptation then used real training pairs plus replay, leaving real development singer/artist groups outside the optimizer. Both results must be considered: controlled generated examples alone cannot demonstrate reliable real-user repair.

In [ ]:
transfer=read(EVIDENCE['presentation_transfer'])
real_rows=[]
for seed,conditions in transfer['results'].items():
    for condition,result in conditions.items():
        for dataset,value in result['by_dataset'].items():
            real_rows.append(dict(stage='frozen transfer',seed=seed,model=condition,dataset=dataset,**value))
for condition,result in ADAPT['results'].items():
    for dataset,value in result['by_dataset'].items():
        real_rows.append(dict(stage='adapted + replay',seed='339',model=condition,dataset=dataset,**value))
real=pd.DataFrame(real_rows)
real['response_pass']=real.median_response_gain>=.25
real['preservation_pass']=real.unshifted_alteration<=1
real['gain_pass']=real.correction_gain>0
real['dataset_gate_pass']=real.response_pass&real.preservation_pass&real.gain_pass
display(real[['stage','seed','model','dataset','records','correction_mae','zero_mae','median_response_gain','unshifted_alteration','outside_alteration','dataset_gate_pass']])
for dataset,part in pd.DataFrame(saved_rows).groupby('dataset'):
    recorded=ADAPT['results']['bpdr']['by_dataset'][dataset]
    for key in ['correction_mae','zero_mae','boundary_sensitivity','unshifted_alteration','outside_alteration']:
        np.testing.assert_allclose(part[key].mean(),recorded[key],rtol=1e-6)
    np.testing.assert_allclose(part.response_gain.median(),recorded['median_response_gain'])
print('Verified adapted BPDR summary against all 304 saved per-variant rows.')
fig,axes=plt.subplots(1,2,figsize=(12,4))
selected=real[(real.model=='bpdr')&(real.seed=='339')]
for ax,dataset in zip(axes,['vocadito','jamendo']):
    part=selected[selected.dataset==dataset]
    ax.bar(part.stage,part.median_response_gain,color=['#ba7456','#35735a'])
    ax.axhline(.25,ls=':',color='grey',label='Engineering threshold')
    ax.set(title=dataset,ylabel='Paired response gain');ax.legend(fontsize=8)
plt.tight_layout();plt.show()
display(pd.DataFrame([dict(model=c,**value) for c,value in M['adaptation_synthetic_retention'].items()]))

The selected adapted model retained useful response on constructed phrases, but response on real-source development examples remained near zero. Small original alteration means the input is preserved; it does not mean the error was repaired. This is a generalization/measurement problem in the tested setup, **not evidence that Vocadito/Jamendo music is bad or musically incorrect**.

## 5. Baselines, alternative placements and later attempts

Compare simple alternatives on the same real development examples, challenge the shortcut of correcting only the longest note, and retain failed real-input/humming candidates. These analyses narrow the claims and guide the next experiment; none supplies an independent final test or listener approval.

In [ ]:
baselines=read(EVIDENCE['presentation_baselines'])
display(pd.DataFrame([dict(model=c,dataset=d,**value) for c,result in baselines['results'].items() for d,value in result['by_dataset'].items()]))
probe=read(EVIDENCE['presentation_probe'])
display(pd.DataFrame([dict(model=c,**value['all']) for c,value in probe['results'].items()]))
print('Above: new constructed phrases whose detuned note is never the longest.')
print('After adaptation, same challenge:')
display(pd.DataFrame([dict(model=c,**value) for c,value in M['adaptation_nonlongest_challenge'].items()]))
print('Real-humming learning receipt:')
humming=read(EVIDENCE['humming_learning'])
display(humming)

## 6. Listen and measure the selected pitch example

This is a **constructed phrase with a known original reference**, processed by the selected adapted model. It demonstrates an audible/measurable local change; it is not a real-human accuracy estimate. Input and output play at the same gain (`normalize=False`). The saved result reader verifies source, candidate and checkpoint hashes before exposing these files.

In [ ]:
from bpdr_input.application.saved_results import pitch_example,noise_example
pitch=pitch_example()
print(pitch['kind']);print(pitch['scope'])
display(pd.DataFrame([{'input_error_cents':abs(pitch['before_cents']),
    'output_error_cents':abs(pitch['after_cents']),
    'single_example_error_reduction':1-abs(pitch['after_cents'])/abs(pitch['before_cents'])}]))
print('Input');display(Audio(filename=str(pitch['input']),normalize=False))
print('Processed preview');display(Audio(filename=str(pitch['output']),normalize=False))
fig,ax=plt.subplots(figsize=(6,3.5))
ax.bar(['Input','Processed preview'],[abs(pitch['before_cents']),abs(pitch['after_cents'])],color=['#a44324','#35735a'])
ax.set(ylabel='Absolute local error (cents)',title='One known-reference example, not aggregate accuracy')
plt.tight_layout();plt.show()

## 7. Real humming plus added rain: a separate noise-processing task

This example starts from a real HumTrans humming recording, mixed with known recorded rain. The deterministic noise preview uses a declared noise-only prefix; **this is not Transformer training or a learned-denoising accuracy score**. The clean-before-mixing recording allows paired SNR measurement. Natural noisy novice recordings need independent evaluation.

In [ ]:
import soundfile as sf
noise=noise_example()
print(noise['kind']);print(noise['scope'])
display(pd.DataFrame([{'added_noise_snr_gain_db':noise['snr_gain_db'],
    'noise_prefix_attenuation_db':noise['reference_attenuation_db'],
    'median_pitch_change_cents':noise['median_pitch_change_cents']}]))
print('Noisy input');display(Audio(filename=str(noise['input']),normalize=False))
print('Processed preview');display(Audio(filename=str(noise['output']),normalize=False))
x,sr=sf.read(noise['input']);y,rate=sf.read(noise['output']);assert sr==rate and x.shape==y.shape
ix=np.arange(0,min(len(x),int(.7*sr)),20)
fig,ax=plt.subplots(figsize=(10,3.5))
ax.plot(ix/sr,x[ix],label='Input noise-only prefix',alpha=.65)
ax.plot(ix/sr,y[ix],label='Processed preview',alpha=.8)
ax.set(xlabel='Seconds',ylabel='Amplitude (same scale)',title='Declared noise-only prefix');ax.legend()
plt.tight_layout();plt.show()

## 8. Runtime behavior, research gates and the next validation

The runtime takes audio alone, estimates features, predicts correction under alternative boundary tracks, and applies bounded permission/render/remeasurement/rollback checks. Identity or a withheld preview is a valid result when an edit lacks support. Original accepted downstream audio remains unchanged; a verified candidate is a separate comparison/download.

Useful next research work is a declared real-singer experiment with better reference/support coverage, a locked independent holdout and seed replication. Collect consented novice humming and clean/noisy pairs, evaluate preservation and correction separately, and add expert/listener review when available. Do not loosen gates merely to force an output difference.

Use the current app's **Input / output** tab for the verified examples; use **Process audio** for new short solo singing/humming. These notebooks explain the evidence and method, while a new live input may be accepted unchanged or have its candidate withheld.

In [ ]:
display(pd.DataFrame([{'gate':k,'recorded_status':v} for k,v in M['gates'].items()]))
display(pd.DataFrame(M['runtime_checks']))
display(pd.DataFrame(M['latency']['rows']))
print('Current research-ready examples:',M['research_ready_examples'])
print('Prepared handoff enabled:',working_card['prepared_handoff_enabled'])
print('Measured runtime hardware:');display(M['latency']['hardware'])
print('Selected checkpoint unchanged:',sha(CHECKPOINT)==working_card['sha256'])
assert sha(CHECKPOINT)==working_card['sha256']
print('Completed evaluation walkthrough. No model fitting or research gate adoption occurred.')